In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Skip malformed rows instead of crashing the load
df = pd.read_csv('products_asos.csv', on_bad_lines='skip')
print(f"Data Loaded: {df.shape[0]} rows and {df.shape[1]} columns")


In [ ]:
# Some prices are strings or empty — coerce forces them to NaN so we can drop cleanly
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

In [ ]:
df['description'] = df['description'].astype(str)

In [ ]:
# Brand is buried in description text like "Coats & Jackets by New Look..."
# We grab the first word after "by" as a raw token
def get_brand(text):
    if isinstance(text, str) and 'by' in text.lower():
        try:
            return text.split('by')[1].strip().split(' ')[0]
        except:
            return 'Unknown'
    return 'Unknown'


In [ ]:
df['brand_raw'] = df['description'].apply(get_brand)

In [ ]:
df.head(3)

In [ ]:
# Multi-word brands get truncated — fix the known ones manually
brand_map = {
    'New': 'New Look',
    'River': 'River Island',
    'Miss': 'Miss Selfridge',
    'TopshopWelcome': 'Topshop'
}
df['Brand'] = df['brand_raw'].replace(brand_map).fillna(df['brand_raw'])

In [ ]:
# Drop brands with ≤5 products — too small to be statistically meaningful
brand_counts = df['brand_raw'].value_counts()
valid_brands = brand_counts[brand_counts > 5].index
df_clean = df[df['Brand'].isin(valid_brands)].copy()
print(df_clean['Brand'].value_counts().head(5))

In [ ]:
# function to analyze stock outs
def calculate_phantom_revenue(size_str):
    if not isinstance(size_str, str):
        return 0, 0.0
    
    # split "UK 6, UK 8 and out of stock" into list
    sizes = size_str.split(',')
    total_sizes = len(sizes)
    
    # count how many items are out of stock
    out_of_stock_count = size_str.count('Out of stock')
    
    # calculate Rate (0.0 to 1.0)
    rate = out_of_stock_count / total_sizes if total_sizes > 0 else 0.0
    
    return out_of_stock_count, rate

In [ ]:
metrics = df_clean['size'].apply(lambda x: calculate_phantom_revenue(x))
df_clean['Stock_Count'] = [x[0] for x in metrics]
df_clean['Stock_Rate'] = [x[1] for x in metrics]

In [ ]:
# Proxy for lost revenue: price × number of OOS size slots
df_clean['Lost_Revenue'] = df_clean['price'] * df_clean['Stock_Count']

In [ ]:
cols = ['Brand', 'name', 'price', 'Stock_Count', 'Stock_Rate', 'Lost_Revenue']
print(df_clean.sort_values(by = 'Lost_Revenue', ascending=False).head(5)[cols])

In [ ]:
# Aggregate to brand level — only keep brands with enough products for signal
brand_strategy = df_clean.groupby('Brand').agg({
    'price': 'mean',
    'Stock_Rate': 'mean',
    'Stock_Count': 'sum',
    'Lost_Revenue': 'sum',
    'name': 'count'
}).reset_index()

brand_strategy = brand_strategy[brand_strategy['name'] > 10]

In [ ]:
# Plot: x=price positioning, y=stockout severity, bubble=revenue at risk

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=brand_strategy, 
    x='price', 
    y='Stock_Rate', 
    size='Lost_Revenue', 
    sizes=(50, 500), 
    alpha=0.7,
    palette='viridis'
)


# Label only the high-risk quadrant (premium price + high stockout)
winners = brand_strategy[
    (brand_strategy['price'] > 40)
    & (brand_strategy['Stock_Rate'] > 0.4)
]

for i in range(len(winners)):
    plt.text(
        winners.iloc[i]['price']+1, 
        winners.iloc[i]['Stock_Rate'], 
        winners.iloc[i]['Brand'],
    )

plt.title('Brand Strategy Analysis')
plt.xlabel('Average Price')
plt.ylabel('Stockout Rate')
plt.axvline(x=40, color='red', linestyle='--')   # price threshold
plt.axhline(y=0.4, color='red', linestyle='--')  # stockout threshold
plt.show()